# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
!pip install -qU requests==2.32.4 langchain-community langchain
!pip install -U langchain langchain-core langchain-text-splitters langchain-community langgraph langgraph-sdk langgraph-prebuilt langchain-classic
!pip install -U langchain-huggingface
!pip install sentence-transformers transformers
!pip install -U -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
  Using cached lan

In [11]:
# YOUR CODE HERE
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
import os

from google.colab import drive
drive.mount('/content/drive')

# 1. Load SOP documents
sop_dir = "corporate_policies/"   # change this to your actual SOP folder

documents = []

for filename in os.listdir(sop_dir):
    if filename.endswith(".md"):
        loader = TextLoader(
            os.path.join(sop_dir, filename),
            encoding="utf-8"
        )
        documents.extend(loader.load())

print(f"Loaded {len(documents)} SOP documents/chunks")

# 2. Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model initialized.")

# 3. Validate embedding using a sample string
sample_embedding = embedding_model.embed_query(
    "Domestic orders deliver within 3-7 business days."
)

print("Embedding vector length:", len(sample_embedding))
print("First 5 values:", sample_embedding[:5])

# 4. Embed the SOP documents
sop_texts = [doc.page_content for doc in documents]
sop_embeddings = embedding_model.embed_documents(sop_texts)

print("Number of embeddings:", len(sop_embeddings))
print("Embedding dimension:", len(sop_embeddings[0]))

# 5. Final validation
assert len(sample_embedding) == 384
assert len(sop_embeddings) == len(documents)
assert all(len(vector) == 384 for vector in sop_embeddings)

print("✓ Embeddings successfully generated.")
print("✓ All embeddings have 384 dimensions.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 13 SOP documents/chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model initialized.
Embedding vector length: 384
First 5 values: [0.012439022772014141, -0.032710589468479156, 0.06439199298620224, -0.004930801689624786, -0.07578545808792114]
Number of embeddings: 13
Embedding dimension: 384
✓ Embeddings successfully generated.
✓ All embeddings have 384 dimensions.


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [4]:
%pip install -U langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetr

In [12]:
# YOUR CODE HERE
from langchain_chroma import Chroma

# Create persistent Chroma vector database
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="/content/drive/MyDrive/Colab Notebooks/HYBRID-RAG-AND-FINE-TUNING-ASSIGNMENT/OUPUT/chroma_db",
    collection_name="sop_documents"
)

print("Chroma vector database created successfully.")

# Validate number of documents in the index
doc_count = vector_db._collection.count()

print("Documents in Chroma index:", doc_count)
print("Expected documents:", len(documents))

# Test semantic retrieval
results = vector_db.similarity_search("refund", k=1)

print("Number of results:", len(results))

if results:
    print("\nRetrieved document:")
    print(results[0].page_content)

    print("\nMetadata:")
    print(results[0].metadata)

Chroma vector database created successfully.
Documents in Chroma index: 13
Expected documents: 13
Number of results: 1

Retrieved document:
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier. Digital
goods and subscription fees already consumed are non-refundable except where
required by local consumer law.

## Refund Methods
Approved refunds are issued to the original payment method. If the original
card or wallet is no longer active, store credit of equal value is offered as
an alternative. Cash refunds are not provided for online orders.

## Processing Time
Once a refund is approved, the amount is released within 5–7 business days.
Bank or card-network posting can add a further 3–5 business days, so advise
customers that the full cycle may take up to 12 business days. Duplicate or
accid

#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [13]:
# YOUR CODE HERE
import os
from google.colab import userdata
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Customer query
query = "I want my money back"

# Naive retrieval: raw query directly into the vector DB
retrieved_docs = vector_db.similarity_search(query, k=1)

# Extract retrieved SOP
context = retrieved_docs[0].page_content

print("Customer Query:")
print(query)

print("\nRetrieved SOP:")
print(context)

print("\nMetadata:")
print(retrieved_docs[0].metadata)

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a customer support assistant.

Answer strictly using this SOP:
{context}

Do not invent information that is not present in the SOP."""
    ),
    ("human", "{query}")
])

formatted_prompt = prompt.invoke({
    "context": context,
    "query": query
})

print(formatted_prompt)

response = llm.invoke(
    formatted_prompt
)

print("Naive RAG Answer:")
print(response.content)

def naive_rag(query, vector_db, llm):
    # Retrieve top-1 document using the raw query
    retrieved_docs = vector_db.similarity_search(query, k=1)

    context = retrieved_docs[0].page_content

    # Build prompt
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """You are a customer support assistant.

Answer strictly using this SOP:
{context}

Do not invent information that is not present in the SOP."""
        ),
        ("human", "{query}")
    ])

    # Generate answer
    response = llm.invoke(
        prompt.invoke({
            "context": context,
            "query": query
        })
    )

    return {
        "query": query,
        "retrieved_document": context,
        "metadata": retrieved_docs[0].metadata,
        "answer": response.content
    }

Enter your OpenAI API Key: ··········
Customer Query:
I want my money back

Retrieved SOP:
# Product Return

## Scope
Covers physically returning an item, whether for a refund, an exchange, or a
warranty replacement. The monetary outcome is governed by the refund policy;
this procedure covers the logistics of getting the item back.

## Return Window
Items may be returned within 30 days of delivery. They should be in resalable
condition with original packaging where practical. Defective or
incorrectly-shipped items can be returned at any reasonable time and never
incur a restocking deduction.

## Return Methods
1. Issue a prepaid return label for defective or wrong-item cases at no cost
   to the customer.
2. For change-of-mind returns, the customer may be responsible for return
   shipping unless a free-returns promotion applies.
3. Provide a return reference and instructions for drop-off or pickup.

## Exchanges
For size, color, or model swaps, create the replacement order on a hold a

---
## Save Artifacts for Downstream Notebooks

In [9]:
# YOUR CODE HERE
CHROMA_DIR = "./chroma_db"

print("Chroma database location:", CHROMA_DIR)
print("Documents indexed:", vector_db._collection.count())

import json

query = "I want my money back"

result = naive_rag(
    query,
    vector_db,
    llm
)

artifact = {
    "query": result["query"],
    "retrieved_document": result["retrieved_document"],
    "metadata": result["metadata"],
    "answer": result["answer"]
}

with open("naive_rag_result.json", "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)

print("✓ Naive RAG result saved to naive_rag_result.json")

test_queries = [
    "I want my money back",
    "Where is my order?",
    "My card was charged twice",
    "How long does delivery take?",
    "I need to cancel my order"
]

results = []

for query in test_queries:
    result = naive_rag(query, vector_db, llm)

    results.append({
        "query": result["query"],
        "retrieved_document": result["retrieved_document"],
        "metadata": result["metadata"],
        "answer": result["answer"]
    })

with open("naive_rag_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(results)} Naive RAG results.")

import os

artifacts = [
    "./chroma_db",
    "./naive_rag_result.json",
    "./naive_rag_results.json"
]

for artifact in artifacts:
    print(
        f"{artifact}:",
        "✓ exists" if os.path.exists(artifact) else "✗ missing"
    )

Chroma database location: ./chroma_db
Documents indexed: 13
✓ Naive RAG result saved to naive_rag_result.json
✓ Saved 5 Naive RAG results.
./chroma_db: ✓ exists
./naive_rag_result.json: ✓ exists
./naive_rag_results.json: ✓ exists


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**